In [10]:
import pandas as pd
import cv2 # Import cv2 directly instead of as cv
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')
%matplotlib inline

import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D, Input

In [11]:
faceProto = "opencv_face_detector.pbtxt"
faceModel = "opencv_face_detector_uint8.pb"
ageProto = "age_deploy.prototxt"
ageModel = "age_net.caffemodel"
genderProto = "gender_deploy.prototxt"
genderModel = "gender_net.caffemodel"

In [20]:
MODEL_MEAN_VALUES = (78.4263377603, 87.7689143744, 114.895847746)
ageList = ['(0-2)', '(4-6)', '(8-12)', '(15-20)', '(25-32)', '(38-43)', '(48-53)', '(60-100)']
genderList = ['Male', 'Female']

In [32]:
import os
import cv2

faceProto = "opencv_face_detector.pbtxt"
faceModel = "opencv_face_detector_uint8.pb"
ageProto = "age_deploy.prototxt"
ageModel = "age_net.caffemodel"
genderProto = "gender_deploy.prototxt"
genderModel = "gender_net.caffemodel"

# Get the current working directory
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")

# Construct the full paths to all model files
age_model_path = os.path.join(current_dir, ageModel)
age_proto_path = os.path.join(current_dir, ageProto)  # Full path for ageProto
gender_model_path = os.path.join(current_dir, genderModel)
gender_proto_path = os.path.join(current_dir, genderProto)  # Full path for genderProto
face_model_path = os.path.join(current_dir, faceModel)
face_proto_path = os.path.join(current_dir, faceProto)  # Full path for faceProto

# Print the paths to verify them:
print(f"Age model path: {age_model_path}")
print(f"Age proto path: {age_proto_path}")
print(f"Gender model path: {gender_model_path}")
print(f"Gender proto path: {gender_proto_path}")
print(f"Face model path: {face_model_path}")
print(f"Face proto path: {face_proto_path}")

# Check if the age model file exists
if os.path.exists(age_model_path) and os.path.exists(age_proto_path): # Also check if age_proto_path exists
    print(f"Age model found at: {age_model_path}")
    ageNet = cv2.dnn.readNetFromCaffe(age_proto_path, age_model_path)  # Use full path for ageProto
else:
    print(f"Error: Age model or prototxt not found.")
    print(f"Age model path: {age_model_path}, exists: {os.path.exists(age_model_path)}")
    print(f"Age proto path: {age_proto_path}, exists: {os.path.exists(age_proto_path)}")
    print("Please check the file paths and ensure the files exist.")

# Load other models using full paths, add similar checks for other models
if os.path.exists(gender_model_path) and os.path.exists(gender_proto_path):
    print(f"Gender model found at: {gender_model_path}")
    genderNet = cv2.dnn.readNetFromCaffe(gender_proto_path, gender_model_path)  # Use full path for genderProto
else:
    print(f"Error: Gender model or prototxt not found.")
    print(f"Gender model path: {gender_model_path}, exists: {os.path.exists(gender_model_path)}")
    print(f"Gender proto path: {gender_proto_path}, exists: {os.path.exists(gender_proto_path)}")
    print("Please check the file paths and ensure the files exist.")

Current working directory: /content
Age model path: /content/age_net.caffemodel
Age proto path: /content/age_deploy.prototxt
Gender model path: /content/gender_net.caffemodel
Gender proto path: /content/gender_deploy.prototxt
Face model path: /content/opencv_face_detector_uint8.pb
Face proto path: /content/opencv_face_detector.pbtxt
Error: Age model or prototxt not found.
Age model path: /content/age_net.caffemodel, exists: False
Age proto path: /content/age_deploy.prototxt, exists: False
Please check the file paths and ensure the files exist.
Error: Gender model or prototxt not found.
Gender model path: /content/gender_net.caffemodel, exists: False
Gender proto path: /content/gender_deploy.prototxt, exists: False
Please check the file paths and ensure the files exist.


In [34]:
cap = cv2.VideoCapture('/content/10_0_0_20161220222308131.jpg.chip.jpg')

In [69]:
import cv2
import time
import os

# ... (your existing code for loading models) ...

# Assuming getFaceBox is defined and requires a faceNet object
# If you are using a pre-trained model from OpenCV's DNN module
# You should create a faceNet object using cv2.dnn.readNet

# Load the face detection model
faceProto = "opencv_face_detector.pbtxt"
faceModel = "opencv_face_detector_uint8.pb"

# **Download the model files if they are not present**
!wget -N https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt -O opencv_face_detector.pbtxt
!wget -N https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel -O opencv_face_detector_uint8.pb

# Assuming your model files are in the same directory as your script
# or you have downloaded them to a specific location, adjust the paths accordingly
face_proto_path = os.path.join(os.getcwd(), faceProto)
face_model_path = os.path.join(os.getcwd(), faceModel)

# Verify if the files exist at the specified paths:
if not os.path.exists(face_proto_path):
    print(f"Error: Prototxt file not found at {face_proto_path}")
if not os.path.exists(face_model_path):
    print(f"Error: Model file not found at {face_model_path}")

faceNet = cv2.dnn.readNetFromCaffe(face_proto_path, face_model_path)

padding = 20

hasFrame, frame = cap.read() # Read the frame outside the loop

if hasFrame:
  frameFace, bboxes = getFaceBox(faceNet, frame) # Process the frame

  cv2.imshow("Detected Faces", frameFace) # Display the processed frame
  cv2.waitKey(0) # Wait indefinitely for a key press
  cv2.destroyAllWindows() # Close the window
else:
  print("Error: Could not read frame from video capture.")

for details.

--2024-11-30 06:08:50--  https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28104 (27K) [text/plain]
Saving to: ‘opencv_face_detector.pbtxt’

opencv_face_detecto 100%[===================>]  27.45K  --.-KB/s    in 0.002s  

2024-11-30 06:08:51 (12.4 MB/s) - ‘opencv_face_detector.pbtxt’ saved [28104/28104]

for details.

--2024-11-30 06:08:51--  https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercont

In [77]:
def getFaceBox(net, frame, conf_threshold=0.7):
    frameOpencvDnn = frame.copy()
    frameHeight = frameOpencvDnn.shape[0]
    frameWidth = frameOpencvDnn.shape[1]
    blob = cv2.dnn.blobFromImage(frameOpencvDnn, 1.0, (300, 300), [104, 117, 123], True, False)

    net.setInput(blob)
    detections = net.forward()
    bboxes = [] # Initialize bboxes list inside the function
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > conf_threshold:
            x1 = int(detections[0, 0, i, 3] * frameWidth)
            y1 = int(detections[0, 0, i, 4] * frameHeight)
            x2 = int(detections[0, 0, i, 5] * frameWidth)
            y2 = int(detections[0, 0, i, 6] * frameHeight)
            bboxes.append([x1, y1, x2, y2])
            cv2.rectangle(frameOpencvDnn, (x1, y1), (x2, y2), (0, 255, 0), int(round(frameHeight / 150)), 8, 0)
    return frameOpencvDnn, bboxes


In [79]:
import cv2
import time
import os

# ... (your existing code for loading models) ...


# Load the face detection model
faceProto = "opencv_face_detector.pbtxt"
faceModel = "opencv_face_detector_uint8.pb"

# **Download the model files if they are not present**
!wget -N https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt -O opencv_face_detector.pbtxt
!wget -N https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel -O opencv_face_detector_uint8.pb

# Assuming your model files are in the same directory as your script
# or you have downloaded them to a specific location, adjust the paths accordingly
face_proto_path = os.path.join(os.getcwd(), faceProto)
face_model_path = os.path.join(os.getcwd(), faceModel)

# Verify if the files exist at the specified paths:
if not os.path.exists(face_proto_path):
    print(f"Error: Prototxt file not found at {face_proto_path}")
if not os.path.exists(face_model_path):
    print(f"Error: Model file not found at {face_model_path}")

faceNet = cv2.dnn.readNetFromCaffe(face_proto_path, face_model_path)

padding = 20

hasFrame, frame = cap.read()  # Read the frame outside the loop

if hasFrame:
    frameFace, bboxes = getFaceBox(faceNet, frame)  # Process the frame and get bboxes

    # Check if any faces were detected
    if not bboxes:
        print("no face Detected, checking next frame")
        # You would likely have a loop here to process the next frame
        # and 'continue' would skip to the next iteration of that loop
    else:
        cv2.imshow("Detected Faces", frameFace)  # Display the processed frame
        cv2.waitKey(0)  # Wait indefinitely for a key press
        cv2.destroyAllWindows()  # Close the window
else:
    print("Error: Could not read frame from video capture.")

for details.

--2024-11-30 06:16:56--  https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28104 (27K) [text/plain]
Saving to: ‘opencv_face_detector.pbtxt’

opencv_face_detecto 100%[===================>]  27.45K  --.-KB/s    in 0.002s  

2024-11-30 06:16:57 (17.1 MB/s) - ‘opencv_face_detector.pbtxt’ saved [28104/28104]

for details.

--2024-11-30 06:16:57--  https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercont

In [84]:
import cv2
import time
import os

# ... (your existing code for loading models) ...


# Load the face detection model
faceProto = "opencv_face_detector.pbtxt"
faceModel = "opencv_face_detector_uint8.pb"

# **Download the model files if they are not present**
!wget -N https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt -O opencv_face_detector.pbtxt
!wget -N https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel -O opencv_face_detector_uint8.pb

# Assuming your model files are in the same directory as your script
# or you have downloaded them to a specific location, adjust the paths accordingly
face_proto_path = os.path.join(os.getcwd(), faceProto)
face_model_path = os.path.join(os.getcwd(), faceModel)

# Verify if the files exist at the specified paths:
if not os.path.exists(face_proto_path):
    print(f"Error: Prototxt file not found at {face_proto_path}")
if not os.path.exists(face_model_path):
    print(f"Error: Model file not found at {face_model_path}")

faceNet = cv2.dnn.readNetFromCaffe(face_proto_path, face_model_path)

padding = 20

hasFrame, frame = cap.read()  # Read the frame outside the loop

if hasFrame:
    frameFace, bboxes = getFaceBox(faceNet, frame)  # Process the frame and get bboxes

    # Check if any faces were detected
    if not bboxes:
        print("no face Detected, checking next frame")
        # You would likely have a loop here to process the next frame
        # and 'continue' would skip to the next iteration of that loop
    else:
        # Iterate through detected bounding boxes within the 'if hasFrame' block
        for bbox in bboxes:
            # print bboxes
            face = frame[max(0, bbox[1] - padding):min(bbox[3] + padding, frame.shape[0] - 1),
                         max(0, bbox[0] - padding):min(bbox[2] + padding, frame.shape[1] - 1)]
            blob = cv2.dnn.blobFromImage(face, 1.0, (227, 227), MODEL_MEAN_VALUES, swapRB=False)  # Assuming MODEL_MEAN_VALUES is defined
            genderNet.setInput(blob)  # Assuming genderNet is defined
            genderPreds = genderNet.forward()
            gender = genderList[genderPreds[0].argmax()]
            ageNet.setInput(blob)  # Assuming ageNet is defined
            agePreds = ageNet.forward()
            age = ageList[agePreds[0].argmax()]

        cv2.imshow("Detected Faces", frameFace)  # Display the processed frame
        cv2.waitKey(0)  # Wait indefinitely for a key press
        cv2.destroyAllWindows()  # Close the window
else:
    print("Error: Could not read frame from video capture.")

for details.

--2024-11-30 06:34:30--  https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28104 (27K) [text/plain]
Saving to: ‘opencv_face_detector.pbtxt’

opencv_face_detecto 100%[===================>]  27.45K  --.-KB/s    in 0.002s  

2024-11-30 06:34:30 (11.6 MB/s) - ‘opencv_face_detector.pbtxt’ saved [28104/28104]

for details.

--2024-11-30 06:34:30--  https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercont

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [85]:
label = "{},{}".format(gender, age)
cv2.putText(frameFace, label, (bbox[0]-5, bbox[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2,cv2.LINE_AA)
cv2.imshow("Age Gender Demo", frameFace)
cv2.waitKey(0)
cv2.destroyAllWindows()

NameError: name 'gender' is not defined

In [87]:
import cv2
import time
import os

# ... (your existing code for loading models) ...


# Load the face detection model
faceProto = "opencv_face_detector.pbtxt"
faceModel = "opencv_face_detector_uint8.pb"

# **Download the model files if they are not present**
!wget -N https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt -O opencv_face_detector.pbtxt
!wget -N https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel -O opencv_face_detector_uint8.pb

# Assuming your model files are in the same directory as your script
# or you have downloaded them to a specific location, adjust the paths accordingly
face_proto_path = os.path.join(os.getcwd(), faceProto)
face_model_path = os.path.join(os.getcwd(), faceModel)

# Verify if the files exist at the specified paths:
if not os.path.exists(face_proto_path):
    print(f"Error: Prototxt file not found at {face_proto_path}")
if not os.path.exists(face_model_path):
    print(f"Error: Model file not found at {face_model_path}")

faceNet = cv2.dnn.readNetFromCaffe(face_proto_path, face_model_path)

padding = 20

hasFrame, frame = cap.read()  # Read the frame outside the loop

if hasFrame:
    frameFace, bboxes = getFaceBox(faceNet, frame)  # Process the frame and get bboxes

    # Check if any faces were detected
    if not bboxes:
        print("no face Detected, checking next frame")
        # You would likely have a loop here to process the next frame
        # and 'continue' would skip to the next iteration of that loop
    else:
        # Iterate through detected bounding boxes within the 'if hasFrame' block
        for bbox in bboxes:
            # print bboxes
            face = frame[max(0, bbox[1] - padding):min(bbox[3] + padding, frame.shape[0] - 1),
                         max(0, bbox[0] - padding):min(bbox[2] + padding, frame.shape[1] - 1)]
            blob = cv2.dnn.blobFromImage(face, 1.0, (227, 227), MODEL_MEAN_VALUES, swapRB=False)  # Assuming MODEL_MEAN_VALUES is defined
            genderNet.setInput(blob)  # Assuming genderNet is defined
            genderPreds = genderNet.forward()
            gender = genderList[genderPreds[0].argmax()] # Assuming genderList is defined
            ageNet.setInput(blob)  # Assuming ageNet is defined
            agePreds = ageNet.forward()
            age = ageList[agePreds[0].argmax()] # Assuming ageList is defined

            # Now that gender and age are calculated, display them on the frame
            label = "{},{}".format(gender, age)
            cv2.putText(frameFace, label, (bbox[0]-5, bbox[1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2,cv2.LINE_AA)

for details.

--2024-11-30 06:39:24--  https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28104 (27K) [text/plain]
Saving to: ‘opencv_face_detector.pbtxt’

opencv_face_detecto 100%[===================>]  27.45K  --.-KB/s    in 0.002s  

2024-11-30 06:39:24 (16.0 MB/s) - ‘opencv_face_detector.pbtxt’ saved [28104/28104]

for details.

--2024-11-30 06:39:24--  https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercont